In [1]:
# --- 1. 安裝 Unsloth (訓練加速神器) ---
print("正在安裝訓練環境，約需 2-3 分鐘...")
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes

# --- 2. 載入模型與資料 ---
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments

# 設定參數
max_seq_length = 2048  # 最長token數
dtype = None  # 自動偵測
load_in_4bit = True  # 壓縮成4-bits

print("正在下載 Llama-3 模型...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

# 加入 LoRA 適配器 (這就是我們要訓練的小腦袋)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,  # LoRA's rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"], # 目標模組
    lora_alpha = 32,  # 縮放係數
    lora_dropout = 0,  # 隨機失活(avoid overfitting)
    bias = "none",  # 不訓練bias
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

# 教材格式 (Alpaca 完美對應我們的 JSONL 結構)
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
{}

### Input:
{}

### Response:
{}"""  # 告訴她只有三個部分instruction input response

EOS_TOKEN = tokenizer.eos_token
def formatting_prompts_func(examples):
    instructions = examples["instruction"]
    inputs       = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, input, output in zip(instructions, inputs, outputs):
        # 這裡會自動把 {"status": "ERR", "sensor": ...} 這串 JSON 當作純文字塞進 Response
        text = alpaca_prompt.format(instruction, input, output) + EOS_TOKEN
        texts.append(text)
    return { "text" : texts, }

# 載入你的 json 檔 (請確認檔案已上傳到左側)
try:

    dataset = load_dataset("json", data_files="semantic_redundancy_train_json.json", split="train")
    dataset = dataset.map(formatting_prompts_func, batched = True)
    print(f"成功載入 {len(dataset)} 筆訓練資料!")
except:
    print("錯誤：找不到 semantic_redundancy_train_json.json，請確認檔案是否上傳成功。")
    exit()

# --- 4. 開始訓練 (Training) ---
print("開始訓練 (Fine-tuning)...")
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 8,
        warmup_ratio = 0.1,
        num_train_epochs = 3,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "outputs_final",
    ),
)

trainer_stats = trainer.train()
print("訓練完成！")

# 轉檔並儲存
print("正在轉檔為 GGUF 格式...")
model.save_pretrained_gguf("llama-3-8b.v6.Q8_0", tokenizer, quantization_method = "q8_0")
print("轉檔完成，可以下載你的 GGUF 模型了！")

正在安裝訓練環境，約需 2-3 分鐘...
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-lw2roq5x/unsloth_c8b3f2ec6e224edca6969526eedb1e6f
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-lw2roq5x/unsloth_c8b3f2ec6e224edca6969526eedb1e6f
  Resolved https://github.com/unslothai/unsloth.git to commit ea017322ddfb6da3a3d11266d77330087f72f063
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 44.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 90.1 M

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-bnb-4bit as a legacy tokenizer.
Unsloth 2026.9.5 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/2500 [00:00<?, ? examples/s]

...

model-00004-of-00004.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [07:10<00:00, 107.67s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [03:32<00:00, 53.01s/it]


Unsloth: Merge process complete. Saved to `/content/llama-3-8b.v6.Q8_0`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF q8_0 might take 3 minutes.
\        /    [2] Single-pass export: converting straight to ['q8_0'] - no separate quantize step.
 "-____-"     In total, you will have to wait at least 6 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b11007-mix-3e83366 (app-b11007-mix-3e83366-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into q8_0 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['llama-3-8b.v6.Q8_0_gguf/llama-3-8b.Q8_0.gguf']
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['llama-3-8b.v6.Q8_0_gguf/l